# 문제 유형 이해와 scikit-learn estimator

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from sklearn.datasets import load_iris

In [ ]:
RANDOM_STATE = 982

# iris 데이터 로딩

In [ ]:
iris = load_iris()
X = iris.data
y_true = iris.target
feature_names = iris.feature_names
target_names = iris.target_names

In [ ]:
df = pd.DataFrame(X, columns=feature_names)
df['target'] = y_true
df['target_name'] = df['target'].map(lambda i: target_names[i])

In [ ]:
df.head()

In [ ]:
print('데이터 크기:', X.shape)
print('클래스 이름:', target_names)
df.describe().T

# iris 데이터 시각화

In [ ]:
sns.set_theme(style='whitegrid', context='notebook')

In [ ]:
sns.pairplot(
    df, hue='target_name', corner=True,
    vars=feature_names,
    kind='scatter')
plt.show()

In [ ]:
sns.pairplot(
    df, hue='target_name', corner=True,
    x_vars=['sepal length (cm)'],
    y_vars=['petal length (cm)'],
    kind='kde')
plt.show()

# Scaling

거리 기반 클러스터링에서는 변수 스케일의 영향이 큽니다. 따라서 실무에서는 대부분 `StandardScaler`를 먼저 적용합니다.

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
scaler = StandardScaler()
scaler.fit(X)
X_scaled = scaler.transform(X)

In [ ]:
df_scaled = pd.DataFrame(X_scaled, columns=feature_names)
df_scaled['target'] = y_true
df_scaled['target_name'] = df['target'].map(lambda i: target_names[i])
df_scaled.head()

In [ ]:
print('데이터 크기:', X_scaled.shape)
print('클래스 이름:', target_names)
df_scaled.describe().T

In [ ]:
sns.pairplot(
    df_scaled, hue='target_name', corner=True,
    vars=feature_names,
    kind='scatter')
plt.show()

# 차원 축소: PCA

In [ ]:
from sklearn.decomposition import PCA

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)
X_pca.shape

In [ ]:
pca_df = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
pca_df['target'] = y_true
pca_df['target_name'] = pca_df['target'].map(lambda i: target_names[i])

In [ ]:
sns.scatterplot(data=pca_df, x='PC1', y='PC2', hue='target_name', s=80)
plt.title('PCA 2D Visualization - True Labels')
plt.show()

# KMeans

가장 기본적으로 먼저 시도하는 클러스터링 알고리즘입니다. Iris는 실제 클래스가 3개이므로 `n_clusters=3`으로 설정합니다.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import confusion_matrix, accuracy_score
from scipy.optimize import linear_sum_assignment

In [ ]:
model_km = KMeans(n_clusters=3, random_state=RANDOM_STATE, init='random', n_init=10) # k-means++
y_kmeans = model_km.fit_predict(X_scaled)

In [ ]:
y_true

In [ ]:
y_kmeans

In [ ]:
# confusion matrix
cm = confusion_matrix(y_true, y_kmeans)

sns.heatmap(cm, annot=True, cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
# Hungarian algorithm
row_ind, col_ind = linear_sum_assignment(-cm)
mapping = {
    pred.item(): true.item()
    for true, pred in zip(row_ind, col_ind)
}
mapping

In [ ]:
# label 변환
y_pred_aligned = np.array([
    mapping[label]
    for label in y_kmeans
])

In [ ]:
acc = accuracy_score(y_true, y_pred_aligned)
print("accuracy =", acc)

In [ ]:
def plot_clusters(title, X_2d, y_pred_aligned):
    # 2차원 좌표에서 클러스터링 결과를 시각화합니다.
    plot_df = pd.DataFrame(X_2d, columns=['x1', 'x2'])
    # plot_df['cluster'] = y_pred.astype(str)

    plot_df['target'] = y_pred_aligned
    plot_df['target_name'] = plot_df['target'].map(lambda i: target_names[i])
    
    plt.figure(figsize=(7, 5))
    sns.scatterplot(data=plot_df, x='x1', y='x2', hue='target_name', s=90, palette='tab10')

    centers = pca.transform(model_km.cluster_centers_)
    centers = pd.DataFrame(centers, columns=["cx", "cy"])
    plt.scatter(
        centers["cx"],
        centers["cy"],
        marker="X",
        s=100,
        label="centers",
    )

    plt.title(title)
    plt.legend(title='Cluster', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.show()

In [ ]:
plot_clusters('KMeans Clustering on PCA Space', X_pca, y_pred_aligned)